In [49]:
import numpy as np

def data_loader():
    X_train = np.load('../datasets/processed/X_train.npy', mmap_mode='r')
    y_train = np.load('../datasets/processed/y_train.npy', mmap_mode='r')
    X_val = np.load('../datasets/processed/X_val.npy', mmap_mode='r')
    y_val = np.load('../datasets/processed/y_val.npy', mmap_mode='r')
    X_test = np.load('../datasets/processed/X_test.npy', mmap_mode='r')
    y_test = np.load('../datasets/processed/y_test.npy', mmap_mode='r')
    classes = np.load('../datasets/processed/classes.npy', allow_pickle=True)
    return X_train, y_train, X_val, y_val, X_test, y_test, classes

In [50]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

def build_cnn_model(input_shape, num_classes):
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling2D(2, 2),

        Conv2D(64, (3, 3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D(2, 2),

        Conv2D(128, (3, 3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D(2, 2),

        Flatten(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    return model

In [51]:
from tensorflow.keras.utils import to_categorical

def preprocess_data(X_train, y_train, X_val, y_val, X_test, y_test):
    X_train = X_train.astype('float32') / 255.0
    X_val   = X_val.astype('float32') / 255.0
    X_test  = X_test.astype('float32') / 255.0

    y_train = to_categorical(y_train)
    y_val   = to_categorical(y_val)
    y_test  = to_categorical(y_test)

    return X_train, y_train, X_val, y_val, X_test, y_test


In [54]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Define callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint('models/best_anime_model.h5', save_best_only=True)
]

# Load and preprocess data
X_train, y_train, X_val, y_val, X_test, y_test, classes = data_loader()
X_train, y_train, X_val, y_val, X_test, y_test = preprocess_data(X_train, y_train, X_val, y_val, X_test, y_test)

# Get input shape and number of classes
input_shape = X_train.shape[1:]  # e.g., (64, 64, 3)
num_classes = y_train.shape[1]   # number of classes after one-hot encoding

# Build and compile the CNN model
model = build_cnn_model(input_shape, num_classes)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

MemoryError: Unable to allocate 1.43 GiB for an array with shape (7789, 128, 128, 3) and data type float32

In [ ]:
# Train the model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    callbacks=callbacks
)

In [ ]:
# Evaluate on test set
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
# Plot accuracy/loss
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend(), plt.title("Loss Curve")
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.legend(), plt.title("Accuracy Curve")
plt.show()